# L10a: Monte Carlo Pricing and Sensitivity Analysis of American Options Premium
Previously, we priced American options on a CRR tree. Can simulation price them? Duration and convexity measured how a bond's price moves with its yield. How much does the premium change when the share price, volatility, time to expiration, or interest rate changes?

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
> * __Estimate and decompose the premium of American options:__ Use the Longstaff–Schwartz method to estimate the premium by Monte Carlo simulation, regressing the value of holding on the share price. Separate option prices into intrinsic value (immediate exercise value) and extrinsic value (time value) to understand what portion of the premium reflects current value versus future potential.
> * __Compute the Greeks for American options:__ Calculate Delta, Theta, Vega, Rho, and Gamma using a local Taylor expansion of the options premium function, and understand how each Greek measures the option price sensitivity to changes in underlying asset price, time to expiration, volatility, and interest rates.
> * __Interpret Delta and compute the probabilities of exercise and profit:__ Apply Delta to determine the number of shares needed to hedge an option position, and compare Delta with the probability that an option will expire in-the-money. Compute the probability of exercise from projected share price distributions at the strike price, and the probability of profit from the same distributions at the breakeven point.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L10a) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Let's compute the premium of an American put by Monte Carlo](CHEME-5660-L10a-Example-LSM-Premium-Fall-2026.ipynb). In this example, we'll work the Longstaff–Schwartz regression by hand on eight paths, then price an American put by simulation and compare it with the CRR lattice and the European premium across strike prices.

> [▶ Let's compute the Greeks for call and put options](CHEME-5660-L10a-Example-Greeks-PutCall-Fall-2026.ipynb). In this example, we compute the Greeks for call and put options using market data. We analyze how the Greeks change with different strike prices and time to expiration, and discuss their implications for options trading strategies.

> [▶ Compute the POP and POE for a short call contract](CHEME-5660-L10a-Example-POE-POP-Call-Fall-2026.ipynb). In this example, we'll compute the probability of profit (POP) and the probability of exercise (POE) for a short call option contract using projected share price distributions and compare with the delta estimate of probability of exercise.

__Review this example on your own.__

> [▶ Let's decompose the premium of a call contract](CHEME-5660-L10a-Example-ExtrinsicValue-Fall-2026.ipynb). In this example, we'll compute the intrinsic and extrinsic values of a call contract, and look at how contract parameters such as the strike price and the days to expiration of the contract influence its extrinsic value.
___

## Company Profile: Susquehanna International Group

[Susquehanna International Group (SIG)](https://sig.com/about/) is a privately held trading firm whose [core business is options trading](https://sig.com/what-we-do/quantitative-trading/). In the early 1980s, a group of friends from college, including Jeff Yass and Arthur Dantchik, began trading options on the floor of the Philadelphia Stock Exchange. They [joined together in 1987](https://sig.com/about/) to start Susquehanna, which is headquartered in Bala Cynwyd, Pennsylvania.

> __What does the firm do?__
>
> * __Options market making:__ A market maker posts prices to buy and sell, so it ends up on the other side of the contracts its customers choose. In October 2026, Susquehanna was the Designated Primary Market Maker for 1,359 of the 5,370 option classes listed at Cboe, more than any other firm ([Cboe symbol directory](https://www.cboe.com/us/options/market_statistics/symbol_reference/?mkt=cone&underlying=1)).
> * __Managing the Greeks:__ The firm's [technical blog](https://sig.com/waves/posts/risk-management-quant-trading/) describes an options book as exposed to Delta, Vega, Theta, and Rho at the same time. The goal is to net these exposures across products, for example by offsetting the positive Vega of calls on one stock with negative Vega on similar stocks.
> * __Training:__ Interns are [trained in options theory](https://sig.com/careers/interns-co-ops/). The firm also [uses poker](https://sig.com/who-we-are/game-theory-decision-science/) to teach new traders decision making under uncertainty: like a trade, a poker hand is a bet whose expected value must be estimated before the outcome is known.

__Explore further:__

* __Jobs and internships:__ Browse Susquehanna's [careers page](https://sig.com/careers/), which lists quantitative trading and research internships, and Discovery Programs for students not yet eligible for an internship.
* __Videos:__ Explore [Susquehanna's YouTube channel](https://www.youtube.com/@LifeatSusquehanna), which includes a 2026 video on no-arbitrage option pricing by Doug Costa, who has led the firm's quantitative research.

__Connection to today's lecture:__ When a customer buys a call from Susquehanna, the firm is short the call and hedges by buying $100\,\Delta_{c}$ shares. As the share price moves, Gamma changes $\Delta_{c}$, so the firm keeps trading shares to stay hedged.

___

## Monte Carlo Pricing of American Options
Previously, we priced European contracts by simulating share prices under the risk-neutral pricing rule $\mathbb{Q}$ and averaging the discounted payoffs. Longstaff and Schwartz extended this approach to American contracts.

__Reference:__
* [Longstaff, F. A.; Schwartz, E. S. (2001). "Valuing American Options by Simulation: A Simple Least-Squares Approach". The Review of Financial Studies. 14 (1): 113–147. doi:10.1093/rfs/14.1.113](https://doi.org/10.1093/rfs/14.1.113)

In this method, we simulate many share price paths under $\mathbb{Q}$, then estimate the expected value of holding at each date by regressing the discounted cash flows from holding on the current share price across paths. Like the lattice, Longstaff and Schwartz work backward from expiration:

__Initialize__: Given the share price $S_{0}$, the strike price $K$, the payoff function $V(K,S)$ of the call or put, the time to expiration $T$ (units: years), the volatility $\sigma$, the risk-free rate $g_{y}$, the number of exercise dates $h$ after today, and the number of paths $M$. Compute the time between exercise dates $\Delta t = T/h$ and the one-step discount factor $\mathcal{D}^{-1}_{t+1,t}(g_{y})=e^{-g_{y}\Delta t}$.

1. __Simulate:__ Generate $M$ share price paths under $\mathbb{Q}$ starting from $S_{0}$ on the dates $t = 0,1,\dots,h$, where $S_{i}(t)$ is the share price on path $i$ at date $t$ and date $h$ is expiration.
2. At expiration, set each path's cash flow to its payoff: $Y_{i}(h) \gets V(K,S_{i}(h))$ for $i = 1,2,\dots,M$, where $Y_{i}(t)$ is path $i$'s cash flow valued at date $t$.
3. For each date $t = h-1, h-2, \dots, 1$ __do__:
   - a. Discount each cash flow back one date: $Y_{i}(t) \gets \mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot Y_{i}(t+1)$.
   - b. __Regress:__ On the in-the-money paths, those with $V(K,S_{i}(t)) > 0$, fit $\hat{C}_{t}(S)=\beta_{0}+\beta_{1}S+\beta_{2}S^{2}$ to the pairs $\bigl(S_{i}(t),Y_{i}(t)\bigr)$ by least squares. The fitted curve estimates the value of holding.
   - c. __Decide:__ For each in-the-money path, if $V(K,S_{i}(t)) > \hat{C}_{t}(S_{i}(t))$, __then__: exercise, replacing the later cash flow with the payoff now, $Y_{i}(t) \gets V(K,S_{i}(t))$.
4. Discount the cash flows to today and average them: $\bar{Y} \gets \mathcal{D}^{-1}_{1,0}(g_{y})\cdot\frac{1}{M}\sum_{i=1}^{M}Y_{i}(1)$.
5. The premium is the larger of exercising today and holding: $\mathcal{P}^{\text{AM}} \gets \max\bigl\{V(K,S_{0}),~\bar{Y}\bigr\}$.

Let's price an American put by simulation and compare it with the lattice and the European premium:

> __Example:__
>
> [▶ Let's compute the premium of an American put by Monte Carlo](CHEME-5660-L10a-Example-LSM-Premium-Fall-2026.ipynb). In this example, we'll work the Longstaff–Schwartz regression by hand on eight paths, then price an American put by simulation and compare it with the CRR lattice and the European premium across strike prices.

Holding on every path gives the European Monte Carlo premium, so the difference estimates the early-exercise premium.
___

## Extrinsic versus Intrinsic Value of American Options
The premium (price) of an American option can be broken down into two components: __intrinsic__ value and __extrinsic__ value.

* __Intrinsic Value__: The intrinsic value of an American option is the __immediate exercise value__. For a call option, it's calculated as $\max(0, S(t) - K)$, where $S(t)$ is the current share price and $K$ is the strike price. For a put option, it's $\max(0, K - S(t))$.
* __Extrinsic Value__: The extrinsic value, also known as time value, represents the additional value of the option based on the potential for future price movements before expiration. It is calculated as the difference between the total option price and its intrinsic value.

The extrinsic value $E_{V}(t)$ can be expressed mathematically as:
$$
\boxed{
\begin{align*}
E_{V}(t) = \text{Option Price} - \text{Intrinsic Value}\quad\blacksquare
\end{align*}}
$$
or in expanded form:
$$
\boxed{
\begin{align*}
E_{V}(t) &= \begin{cases}
\mathcal{P}_{c}(K,S(t)) - \max(0, S(t) - K) & \text{(for Call Options)} \\
\mathcal{P}_{p}(K,S(t)) - \max(0, K - S(t)) & \text{(for Put Options)}
\end{cases}
\end{align*}}
$$
This creates an interesting edge case: when the option is __out of the money__, i.e., the option has no intrinsic value, the entire option price is extrinsic value.

> __Why do people pay money for something that is "worthless"?__ The answer lies in the potential for future price movements. Even if an option is currently out of the money, there is still a chance that the underlying asset's price could move favorably before expiration, making the option valuable.

As time moves towards expiration, the extrinsic value of an option tends to decrease, a phenomenon known as __time decay__. This is because there is less time for the underlying asset's price to move in a favorable direction.

> __Review on your own:__
>
> [▶ Let's decompose the premium of a call contract](CHEME-5660-L10a-Example-ExtrinsicValue-Fall-2026.ipynb). In this example, we'll compute the intrinsic and extrinsic values of a call contract, and look at how contract parameters such as the strike price and the days to expiration of the contract influence its extrinsic value.

Let's look at some option contracts in the market and discuss their intrinsic and extrinsic values:

<div>
    <center>
        <img src="figs/CHEME-5660-L10a-Example-ExtrinsicValue-Fall-2026.svg" width="600" alt="Call premium against strike price for five times to expiration, from the shortest to the longest, with the intrinsic value as a red dashed line and the current share price as a gray dashed vertical line. Every premium curve lies on or above the intrinsic value, and the gap is largest near the current share price and grows with the time to expiration."/>
    </center>
</div>

__Hmmmm:__ We see that the price of the call contract changes with time to expiration and the strike price (relative to the share price of the underlying asset), so how does the premium change with other factors?

___

## Options Premiums and Market Conditions
The central question we will address in this lecture is: "How do American options premiums change with market conditions?" To answer this, we will explore the sensitivity of American options premiums to various factors, including underlying asset price, volatility, time to expiration, and interest rates (risk-free rate).

This question is answered (at least locally) by computing sensitivity coefficients of the options premium collectively known as the __Greeks.__

> __The Greeks__ 
> 
> The Greeks are a set of (local) measures that describe how the price of an option changes in response to changes in various parameters. They are essential tools for options traders to assess and manage the risks associated with options positions.
> - **Delta ($\Delta$)**: Measures the sensitivity of the option's price to a `+1 USD/share` change in the underlying asset's price. However, $\Delta$ also represents the hedge ratio, i.e., the number of shares of the underlying asset needed to hedge one option contract, and is often used to estimate the probability that the option will finish in-the-money at expiration.
> - **Theta ($\Theta$)**: Measures the sensitivity of the option's price to a `1-day decrease` in the lifetime of the contract (time decay). Theta benefits option sellers, as options lose value as expiration approaches, all else being equal.
> - **Vega ($\nu$)**: Measures the sensitivity of the option's price to a `+1%` change in the volatility of the underlying asset. The higher the volatility, the higher the option premium, as there is a greater chance of significant price movements.
> - **Gamma ($\Gamma$)**: Measures the sensitivity of $\Delta$ to a `+1 USD/share` change in the underlying asset's price. This is the second derivative of the option price with respect to the underlying asset price.
> - **Rho ($\rho$)**: Measures the sensitivity of the option's price to a `+1%` change in the risk-free interest rate. This is generally a minor effect for most options, but can be more significant for longer-term options.
>
> When trading, most trading platforms will compute these for you automatically!

### Derivation
Suppose we have an American option contract of type $\star\in\left\{c,p\right\}$, written against an underlying asset with share price $S$ and (implied) volatility $\sigma$, that expires at time $T$. At calendar time $t$ (in years), the premium of that contract is $\mathcal{P}_{\star} = \mathcal{P}_{\star}(S, \sigma, t, g_{y})$, where $g_{y}$ is the risk-free interest rate. For small changes in these inputs, a second-order Taylor expansion (second order in $S$, first order in the others) gives:
$$
\begin{align*}
\Delta\mathcal{P}_{\star} &\approx \underbrace{\left(\frac{\partial \mathcal{P}_{\star}}{\partial S}\right)}_{=\Delta_{\star}}\;\Delta S + 
\underbrace{\left(\frac{\partial \mathcal{P}_{\star}}{\partial \sigma}\right)}_{=\nu_{\star}}\;\Delta\sigma + 
\underbrace{\left(\frac{\partial \mathcal{P}_{\star}}{\partial t}\right)}_{=\Theta_{\star}}\;\Delta t + \underbrace{\left(\frac{\partial \mathcal{P}_{\star}}{\partial g_{y}}\right)}_{=\rho_{\star}}\;\Delta g_{y} + \frac{1}{2}\underbrace{\left(\frac{\partial^{2} \mathcal{P}_{\star}}{\partial S^{2}}\right)}_{=\Gamma_{\star}}\;(\Delta S)^{2}\\
& = \Delta_{\star}\;\Delta S + \nu_{\star}\;\Delta\sigma + \Theta_{\star}\;\Delta t + \rho_{\star}\;\Delta g_{y} + \frac{1}{2}\;\Gamma_{\star}\;(\Delta S)^{2}
\end{align*}
$$
where all the partial derivatives are evaluated at the current market conditions, i.e., the current values of the parameters: $S$, $\sigma$, $t$, and $g_{y}$. In this expansion, $\Delta t$ is in years, and $\Delta\sigma$ and $\Delta g_{y}$ are decimals. Trading platforms quote $\Theta_{\star}/365$ per day, and $\nu_{\star}/100$ and $\rho_{\star}/100$ per percentage point.

$\Delta_{\star}$ and $\Gamma_{\star}$ play the roles of a bond's duration and convexity: they are the slope and the curvature of the premium as a function of the share price.

> __Example:__ Suppose a long call has $\Delta_{c}=0.50$ and $\Gamma_{c}=0.02$ per USD/share. If the share price rises by $\Delta S=5$ USD/share, the Delta term is $0.50\times5=2.50$ USD/share, and the Gamma term adds $\tfrac{1}{2}(0.02)(5)^{2}=0.25$ USD/share, so the premium rises by about 2.75 USD/share. If the share price falls by 5 USD/share, the estimate is $-2.50+0.25=-2.25$ USD/share. The Gamma term adds to the premium whichever way the share price moves.

### Computing the Greeks of American Options
We compute the American premium by backward induction on the CRR tree, so there is no formula to differentiate. Instead, we estimate each Greek by repricing the contract on the tree with one input moved up and down by a step $\epsilon$, holding the other inputs fixed:
$$
\begin{align*}
\frac{\partial\mathcal{P}_{\star}}{\partial x} & \approx \frac{\mathcal{P}_{\star}(x+\epsilon)-\mathcal{P}_{\star}(x-\epsilon)}{2\epsilon}\qquad x\in\left\{S,\sigma,t,g_{y}\right\}\\
\Gamma_{\star} & \approx \frac{\mathcal{P}_{\star}(S+\epsilon)-2\,\mathcal{P}_{\star}(S)+\mathcal{P}_{\star}(S-\epsilon)}{\epsilon^{2}}
\end{align*}
$$
Let's look at an example where we compute and analyze the Greeks for call and put options.

> __Example:__
> 
> [▶ Let's compute the Greeks for call and put options](CHEME-5660-L10a-Example-Greeks-PutCall-Fall-2026.ipynb). In this example, we compute the Greeks for call and put options using market data. We analyze how the Greeks change with different strike prices and time to expiration, and discuss their implications for options trading strategies.
___

## The Secret Lives of Delta $\Delta$
The $\Delta$ of an option measures how much the price of the option changes for a `+1 USD/share` change in the price of the underlying asset. However, it has two other important interpretations:

> __Delta Interpretations:__
> 
> * __Hedging Ratio:__ Delta can be interpreted as the number of shares of the underlying asset needed to hedge one option contract. For example, if an option contract has $\Delta$, it means that for every $n$ contracts held ($n<0$ if the contracts are sold), an investor would need to hold $-{100}\cdot{n}\cdot\Delta$ shares of the underlying asset to be delta-neutral. A delta-neutral position is one where the overall portfolio value is insensitive to small changes in the price of the underlying asset.
> * __Probability of Exercise:__ Delta is also often used as an estimate of the probability that the option will expire in-the-money (ITM). For example, a delta of 0.7 for a call option suggests a 70% chance that the share price at expiration will be at least one cent above the strike price.

We've already seen an example of the hedge ratio interpretation of $\Delta$ in the last example notebook. Let's look at the probability interpretation of $\Delta$. In particular, we can compare $\Delta$ with the probability of exercise:

> __Probability of Exercise (POE):__
>
> The probability of exercise (POE) is the probability that at the expiration of the (composite or single) contract at time $t = T$, the option will finish in-the-money, i.e.,
> $$
> \boxed{
> \begin{align*}
> \text{POE} & = \begin{cases}
> \mathbb{P}\left(S_{T} > K + 0.01\right) & \text{(for Call Options)} \\
> \mathbb{P}\left(S_{T} < K - 0.01\right) & \text{(for Put Options)}
> \end{cases}
> \end{align*}}
> $$
> where $S_{T}$ is the share price of the underlying asset at expiration and $K$ is the strike price. In the Black–Scholes–Merton model, a European call has $\Delta_{c}=N(d_{+})$, while the risk-neutral probability that it finishes in-the-money is $N(d_{-})$, which is smaller. We'll check the conjecture that $\Delta \approx \text{POE}$ using an example.

We can also compute the __Probability of Profit (POP)__ for an option contract, which is closely related to the probability of exercise.

> __Probability of Profit (POP):__ 
>
> The probability of profit (POP) is the probability that at the expiration of the (composite or single) contract at time $t = T$, the profit of the contract is positive, i.e.,
> $$
> \boxed{
> \begin{align*}
> \text{POP} & = \mathbb{P}\left(\hat{P}_{\mathcal{C}} > 0\right)
> \end{align*}}
> $$
> where $\hat{P}_{\mathcal{C}}$ is the profit of the (single or composite) contract at expiration. 

Let's look at an example of a call option contract to compute the POE and POP.

> __Example:__
> 
> [▶ Compute the POP and POE for a short call contract](CHEME-5660-L10a-Example-POE-POP-Call-Fall-2026.ipynb). In this example, we'll compute the probability of profit (POP) and the probability of exercise (POE) for a short call option contract using projected share price distributions and compare with the delta estimate of probability of exercise.
___

## Summary
This lecture prices American options by Monte Carlo simulation, splits the premium into intrinsic and extrinsic value, and introduces the Greeks as sensitivity measures that quantify how American options premiums respond to changes in market conditions.

> __Key Takeaways:__
> 
> * __Estimating and decomposing the American premium:__ Longstaff–Schwartz works backward along simulated paths, regressing the value of holding on the share price. Holding on every path gives the European premium. Intrinsic value represents the immediate exercise value, while extrinsic value (time value) captures the potential for favorable price movement before expiration. Time to expiration, volatility, and moneyness (relationship between stock price and strike price) all influence how much extrinsic value an option possesses, with longer-dated options typically carrying more extrinsic value than shorter-dated ones.
> * __The Greeks quantify option price sensitivity:__ Delta measures sensitivity to underlying price changes, Theta captures time decay, Vega measures volatility sensitivity, Rho tracks interest rate sensitivity, and Gamma measures the rate of change in Delta. These measures are the partial derivatives in a local Taylor expansion of the options premium, which we estimate for American options by repricing the CRR tree.
> * __Delta serves three distinct purposes:__ Delta represents the rate of change in option price per dollar change in underlying price, determines the hedge ratio for constructing delta-neutral positions, and is often used to approximate the probability that the option will expire in-the-money. We checked that approximation against the probability of exercise (POE), computed from a projected share price distribution at the strike price. The probability of profit (POP) uses the same distribution at the breakeven point to estimate whether the position will generate a net profit at expiration.

Understanding the Greeks and probability measures enables assessment of option positions and their exposure to different market risk factors.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___